In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =========================
# 1. Imports
# =========================
import os
import json
import numpy as np
import pandas as pd
from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# =========================
# 2. LOAD PRIMEVUL JSONL DATA
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/svm-primevul26"

def load_jsonl(file):
    X, y = [],[]
    # Added error handling in case exact filenames differ slightly
    if not os.path.exists(file):
        raise FileNotFoundError(f"❌ File not found: {file}")
        
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            
            # PrimeVul uses various keys depending on the specific pairing
            code = obj.get("func_before") or obj.get("code") or obj.get("func") or ""
            label = obj.get("target", obj.get("label", 0))

            if code.strip():
                X.append(code)
                y.append(int(label))
                
    return X, y

print("Loading PrimeVul datasets...")
train_file = os.path.join(base_path, "primevul_train_paired.jsonl")
val_file   = os.path.join(base_path, "primevul_valid_paired.jsonl")
test_file  = os.path.join(base_path, "primevul_test_paired.jsonl")

X_train, y_train = load_jsonl(train_file)
X_val,   y_val   = load_jsonl(val_file)
X_test,  y_test  = load_jsonl(test_file)

# We keep them STRICTLY separated to prevent Data Leakage!
print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

# =========================
# 3. DUAL FEATURE ENGINEERING (Word + Char)
# =========================
print("\n⚙️ Vectorizing Data (Dual Word+Char N-Grams)...")

# 3A. Word-Level Vectorizer (Captures Exact Syntax & Keywords)
vec_word = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    token_pattern=r'[a-zA-Z0-9_]+|[^\w\s]', # Captures punctuation/operators like ==, ->, {
    dtype=np.float32
)

# 3B. Char-Level Vectorizer (Captures Motifs, typos, & Sub-structures)
vec_char = TfidfVectorizer(
    max_features=15000,
    analyzer='char_wb', # Character chunks strictly inside words
    ngram_range=(3, 5), 
    min_df=3,
    max_df=0.90,
    dtype=np.float32
)

# Fit & Transform Train Set
print("Fitting vectorizers to Training data...")
X_tr_w = vec_word.fit_transform(X_train)
X_tr_c = vec_char.fit_transform(X_train)

# Stack them horizontally into a single rich feature matrix
X_train_vec = hstack([X_tr_w, X_tr_c]).tocsr() 

# Transform Val & Test Sets (Never fit on these!)
X_val_vec  = hstack([vec_word.transform(X_val), vec_char.transform(X_val)]).tocsr()
X_test_vec = hstack([vec_word.transform(X_test), vec_char.transform(X_test)]).tocsr()

print(f"Total Combined Features: {X_train_vec.shape[1]}")

# =========================
# 4. SVM MODEL (LINEAR & CALIBRATED)
# =========================
print("\n🚀 Training & Calibrating SVM (This may take a moment)...")

# C=0.5 slightly regularizes the model to prevent overfitting on the massive 30k feature space
svm = LinearSVC(C=0.5, random_state=42)

# CalibratedClassifierCV translates raw SVM distances into accurate probabilities 
model = CalibratedClassifierCV(svm, method='sigmoid', cv=5, n_jobs=-1)
model.fit(X_train_vec, y_train)

# =========================
# 5. MACRO-F1 THRESHOLD TUNING (ON VAL SET)
# =========================
val_probs = model.predict_proba(X_val_vec)[:, 1]

best_macro_f1 = 0
best_t = 0.5

print("\n🔍 Threshold tuning strictly on Validation Set:")

# Fast mathematical loop using confusion_matrix
for t in np.arange(0.20, 0.82, 0.02):
    preds = (val_probs > t).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
    
    # Calculate True Negative metrics
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1_0 = 2 * (precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0

    # Calculate True Positive metrics
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_1 = 2 * (precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0

    # We optimize for Macro F1 to balance Class 0 and Class 1 perfectly
    macro_f1 = (f1_0 + f1_1) / 2

    # Print cleanly (every 10th step)
    if round(t * 100) % 10 == 0:
        print(f"t={t:.2f} → Macro_F1={macro_f1:.4f} | R0={recall_0:.2f}, R1={recall_1:.2f}")

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_t = t

print(f"\nBest threshold found: {best_t:.2f}")

# =========================
# 6. FINAL EVALUATION (ON TEST SET)
# =========================
print("\nEvaluating untouched Test Set...")
test_probs = model.predict_proba(X_test_vec)[:, 1]
final_preds = (test_probs > best_t).astype(int)

acc = accuracy_score(y_test, final_preds)
report = classification_report(y_test, final_preds, output_dict=True)

print("\n✅ Accuracy:", acc)
print("\n📊 Classification Report:\n", classification_report(y_test, final_preds))

# =========================
# 7. SAVE RESULTS
# =========================
# Safe extraction of Class 1 label key (handles string vs int typing in sklearn)
label_key = "1" if "1" in report else [k for k in report.keys() if str(k).startswith("1")][0]

results = {
    "Model": "SVM_PrimeVUL26",
    "Dataset": "PrimeVul",
    "Accuracy": acc,
    "Precision_vuln": report[label_key]['precision'],
    "Recall_vuln": report[label_key]['recall'],
    "F1_vuln": report[label_key]['f1-score'],
    "Macro_F1": report['macro avg']['f1-score'],
    "Best_threshold": best_t
}

# Saved exactly as requested
output_csv_path = "/kaggle/working/SVM_PrimeVUL26_results.csv"
pd.DataFrame([results]).to_csv(output_csv_path, index=False)

print(f"\n✅ Results saved to {output_csv_path}")

Loading PrimeVul datasets...
Train size: 7578 | Val size: 960 | Test size: 870

⚙️ Vectorizing Data (Dual Word+Char N-Grams)...
Fitting vectorizers to Training data...
Total Combined Features: 30000

🚀 Training & Calibrating SVM (This may take a moment)...

🔍 Threshold tuning strictly on Validation Set:
t=0.20 → Macro_F1=0.3333 | R0=0.00, R1=1.00
t=0.30 → Macro_F1=0.3333 | R0=0.00, R1=1.00
t=0.40 → Macro_F1=0.3393 | R0=0.01, R1=1.00
t=0.50 → Macro_F1=0.5417 | R0=0.54, R1=0.54
t=0.60 → Macro_F1=0.3356 | R0=1.00, R1=0.00
t=0.70 → Macro_F1=0.3333 | R0=1.00, R1=0.00
t=0.80 → Macro_F1=0.3333 | R0=1.00, R1=0.00

Best threshold found: 0.50

Evaluating untouched Test Set...

✅ Accuracy: 0.539080459770115

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.54      0.55      0.55       435
           1       0.54      0.52      0.53       435

    accuracy                           0.54       870
   macro avg       0.54      0.54      0.54      